# PCA of lab IMU embeddings

This notebook runs separate PCA analyses on generic and stroke-adapted LIMU-BERT-X and HART embeddings. Each row is one participant, visit, and NW walking trial. Window embeddings were mean-pooled within each trial before PCA, giving 77 trial rows from 17 participants. Fall-risk labels are shown only as plot annotations and are not PCA inputs.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from IPython.display import display

search_paths = [Path.cwd(), *Path.cwd().parents]
repo_root = next((p for p in search_paths if (p / 'Data/analysis/master_analysis_table_pca.csv').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Could not find Data/analysis/master_analysis_table_pca.csv from the current directory.')

df = pd.read_csv(repo_root / 'Data/analysis/master_analysis_table_pca.csv')
results_dir = repo_root / 'Data/analysis/pca_results'
results_dir.mkdir(parents=True, exist_ok=True)
print(f'Loaded {len(df)} trial rows from {df.participant_id.nunique()} participants.')

In [ ]:
feature_blocks = {
    'LIMU-BERT-X generic': [c for c in df.columns if c.startswith('limu_generic_embedding_')],
    'LIMU-BERT-X stroke-adapted': [c for c in df.columns if c.startswith('limu_stroke_adapted_embedding_')],
    'HART generic': [c for c in df.columns if c.startswith('hart_generic_embedding_')],
    'HART stroke-adapted': [c for c in df.columns if c.startswith('hart_stroke_adapted_embedding_')],
}
for name, columns in feature_blocks.items():
    print(f'{name}: {len(columns)} embedding dimensions')

In [ ]:
def fit_embedding_pca(name, columns):
    values = df[columns].apply(pd.to_numeric, errors='coerce')
    keep = values.notna().all(axis=1)
    values = values.loc[keep]
    scaler = StandardScaler()
    scaled = scaler.fit_transform(values)
    pca = PCA(svd_solver='full')
    scores = pca.fit_transform(scaled)
    score_df = pd.DataFrame(scores[:, :4], columns=['PC1', 'PC2', 'PC3', 'PC4'], index=values.index)
    score_df['participant_id'] = df.loc[keep, 'participant_id'].to_numpy()
    score_df['fall_risk_label'] = df.loc[keep, 'fall_risk_label'].to_numpy()
    score_df['walk_trial_name'] = df.loc[keep, 'walk_trial_name'].to_numpy()
    cumulative = np.cumsum(pca.explained_variance_ratio_)
    summary = {
        'embedding_block': name,
        'trial_rows': len(values),
        'participants': df.loc[keep, 'participant_id'].nunique(),
        'embedding_dimensions': len(columns),
        'PC1_variance_pct': 100 * pca.explained_variance_ratio_[0],
        'PC2_variance_pct': 100 * pca.explained_variance_ratio_[1],
        'PC1_PC2_cumulative_pct': 100 * cumulative[1],
        'components_for_80pct': int(np.searchsorted(cumulative, .80) + 1),
        'components_for_90pct': int(np.searchsorted(cumulative, .90) + 1),
    }
    return pca, score_df, summary

pca_models = {}
score_tables = {}
summary_rows = []
for name, columns in feature_blocks.items():
    pca_models[name], score_tables[name], summary = fit_embedding_pca(name, columns)
    summary_rows.append(summary)

summary_table = pd.DataFrame(summary_rows)
summary_table.to_csv(results_dir / 'explained_variance_summary.csv', index=False)
display(summary_table.round(2))

In [ ]:
def show_variance_curve(name):
    ratios = pca_models[name].explained_variance_ratio_
    cumulative = np.cumsum(ratios)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(np.arange(1, len(cumulative) + 1), 100 * cumulative, marker='.', linewidth=1)
    ax.axhline(80, color='gray', linestyle='--', linewidth=1, label='80%')
    ax.axhline(90, color='gray', linestyle=':', linewidth=1, label='90%')
    ax.set(xlabel='Number of principal components', ylabel='Cumulative explained variance (%)',
           title=f'{name}: cumulative explained variance', ylim=(0, 101))
    ax.grid(alpha=.2)
    ax.legend()
    fig.tight_layout()
    slug=name.lower().replace(' ', '_').replace('-', '_')
    fig.savefig(results_dir / f'{slug}_variance.png', dpi=160)
    plt.show()

def show_score_plot(name):
    scores=score_tables[name]
    labels=scores['fall_risk_label'].fillna('Unclassified')
    fig, ax=plt.subplots(figsize=(7, 5))
    for label in sorted(labels.unique()):
        mask=labels==label
        ax.scatter(scores.loc[mask,'PC1'],scores.loc[mask,'PC2'],label=label,alpha=.8,s=38)
    pca=pca_models[name]
    ax.set(xlabel=f'PC1 ({100*pca.explained_variance_ratio_[0]:.1f}%)',
           ylabel=f'PC2 ({100*pca.explained_variance_ratio_[1]:.1f}%)',
           title=f'{name}: trial-level scores')
    ax.legend(title='Fall-risk label',fontsize=8)
    ax.grid(alpha=.2)
    fig.tight_layout()
    slug=name.lower().replace(' ', '_').replace('-', '_')
    fig.savefig(results_dir / f'{slug}_pc1_pc2.png',dpi=160)
    plt.show()

## LIMU-BERT-X generic

Cumulative variance explained, shown separately for this embedding block.

In [ ]:
show_variance_curve('LIMU-BERT-X generic')

PC1 and PC2 trial scores. Color indicates the participant-level fall-risk label; it was not used to fit PCA.

In [ ]:
show_score_plot('LIMU-BERT-X generic')

## LIMU-BERT-X stroke-adapted

Cumulative variance explained, shown separately for this embedding block.

In [ ]:
show_variance_curve('LIMU-BERT-X stroke-adapted')

PC1 and PC2 trial scores. Color indicates the participant-level fall-risk label; it was not used to fit PCA.

In [ ]:
show_score_plot('LIMU-BERT-X stroke-adapted')

## HART generic

Cumulative variance explained, shown separately for this embedding block.

In [ ]:
show_variance_curve('HART generic')

PC1 and PC2 trial scores. Color indicates the participant-level fall-risk label; it was not used to fit PCA.

In [ ]:
show_score_plot('HART generic')

## HART stroke-adapted

Cumulative variance explained, shown separately for this embedding block.

In [ ]:
show_variance_curve('HART stroke-adapted')

PC1 and PC2 trial scores. Color indicates the participant-level fall-risk label; it was not used to fit PCA.

In [ ]:
show_score_plot('HART stroke-adapted')

## Participant-level correlations with full-signal gait measures

The gait targets come from `feature_dataset_lab/full_signal/XSENS/xsens_features.csv`, matched to the NW trial embeddings by participant, visit, and trial. For each participant, PC scores and gait measures are averaged over available NW trials; Spearman correlations are calculated across participants. This avoids treating repeat trials as independent people.

The source contains nine `sacrum_walk_duration_s` values of 655.35 seconds, which are invalid sentinel values. For these trials, duration and its derived cadence and step-time values are set to missing for this analysis. Step counts are retained. Benjamini–Hochberg adjusted q-values account for the 64 exploratory PC-by-measure comparisons.

In [ ]:
from scipy.stats import spearmanr

pc_names = ['PC1', 'PC2', 'PC3', 'PC4']
gait_targets = {
    'Step count': 'sacrum_walk_step_count',
    'Cadence (steps/min)': 'sacrum_walk_cadence_steps_min',
    'Mean step time (s)': 'sacrum_walk_mean_step_time_s',
    'Walking duration (s)': 'sacrum_walk_duration_s',
}
gait_data = df.copy()
invalid_duration = np.isclose(gait_data['sacrum_walk_duration_s'], 655.35)
for col in ['sacrum_walk_duration_s', 'sacrum_walk_cadence_steps_min', 'sacrum_walk_mean_step_time_s']:
    gait_data.loc[invalid_duration, col] = np.nan
print(f'Excluded duration-derived values for {int(invalid_duration.sum())} sentinel trials; step counts retained.')

participant_gait = gait_data.groupby('participant_id')[list(gait_targets.values())].mean()
rows = []
for block_name, scores in score_tables.items():
    participant_pc = scores.groupby('participant_id')[pc_names].mean()
    aligned = participant_pc.join(participant_gait, how='inner')
    for pc in pc_names:
        for measure_label, measure_col in gait_targets.items():
            pair = aligned[[pc, measure_col]].dropna()
            rho, p_value = spearmanr(pair[pc], pair[measure_col])
            rows.append({
                'embedding_block': block_name,
                'PC': pc,
                'gait_measure': measure_label,
                'n_participants': len(pair),
                'spearman_rho': rho,
                'p_value': p_value,
            })
correlations = pd.DataFrame(rows)
# Benjamini-Hochberg adjustment across all 64 exploratory correlations.
valid_p = correlations['p_value'].notna().to_numpy()
order_local = np.argsort(correlations.loc[valid_p, 'p_value'].to_numpy())
valid_indices = np.flatnonzero(valid_p)
p_sorted = correlations.loc[valid_p, 'p_value'].to_numpy()[order_local]
m = len(p_sorted)
q_sorted = np.minimum.accumulate((p_sorted * m / np.arange(1, m + 1))[::-1])[::-1]
q_values = np.full(len(correlations), np.nan)
q_values[valid_indices[order_local]] = np.clip(q_sorted, 0, 1)
correlations['fdr_q_value'] = q_values
correlations.to_csv(results_dir / 'pca_pc_gait_correlations_subject_level.csv', index=False)
print(f'Correlation sample size by gait measure: {correlations.groupby("gait_measure").n_participants.min().to_dict()}')

### LIMU-BERT-X generic

In [ ]:
display(correlations.loc[correlations.embedding_block == 'LIMU-BERT-X generic', ['PC', 'gait_measure', 'n_participants', 'spearman_rho', 'p_value', 'fdr_q_value']].round(3).reset_index(drop=True))

### LIMU-BERT-X stroke-adapted

In [ ]:
display(correlations.loc[correlations.embedding_block == 'LIMU-BERT-X stroke-adapted', ['PC', 'gait_measure', 'n_participants', 'spearman_rho', 'p_value', 'fdr_q_value']].round(3).reset_index(drop=True))

### HART generic

In [ ]:
display(correlations.loc[correlations.embedding_block == 'HART generic', ['PC', 'gait_measure', 'n_participants', 'spearman_rho', 'p_value', 'fdr_q_value']].round(3).reset_index(drop=True))

### HART stroke-adapted

In [ ]:
display(correlations.loc[correlations.embedding_block == 'HART stroke-adapted', ['PC', 'gait_measure', 'n_participants', 'spearman_rho', 'p_value', 'fdr_q_value']].round(3).reset_index(drop=True))

## Interpretation notes

- PCA uses trial-level mean-pooled embeddings; correlations use participant-level averages across NW trials.
- Fall-risk labels only annotate the PCA score plots and are not used in PCA or these correlations.
- The source includes invalid sentinel durations. The notebook masks those durations and the cadence and step-time values derived from them, while retaining step counts.
- Step count, cadence, step time, and duration are mathematically related. Interpret them as related measures, not independent confirmations.
- These correlations are exploratory with 17 participants. They do not establish clinical validity or predictive performance.